
# EDA - Online Retail II

El objetivo de este EDA es entender los datos que tenemos disponibles, ya que la empresa posee un historial de compras rico, pero aún podemos utilizar esa información para sugerir productos relevantes para cada cliente, en lugar de hacer sugerencias genéricas que no generen una conexión real. De esta forma esperamos aprovechar mejor las oportunidades de recompra y venta cruzada, con el fin de aumentar el ticket promedio y las ganancias.

Con esto en mente, esperamos que nuestro modelo pueda recibir la información de un cliente y devolver algo como:

1. Producto A
2. Producto B
3. Producto C
4. Producto D ...

Después evaluaremos si estos productos coinciden con futuras compras del cliente. El objetivo entonces de nuestro modelo será **"Predecir y jerarquizar productos que el cliente podría comprar posteriormente"**.



## Entendimiento del negocio

Comenzaremos por obtener un contexto real del negocio:

1. **¿A qué se dedica la empresa y qué información contiene el dataset?**  
   La empresa se dedica principalmente a vender productos y regalos para distintas ocasiones mediante comercio electrónico. El dataset contiene el historial de transacciones realizadas entre diciembre de 2009 y diciembre de 2011 e incluye información sobre facturas, productos, cantidades, fechas, precios, clientes y países.

2. **¿Qué problema enfrenta la empresa por no contar con recomendaciones personalizadas?**  
   La empresa dispone de información suficiente para conocer parte del comportamiento de compra de sus clientes, pero no cuenta con un mecanismo que convierta ese historial en recomendaciones personalizadas. Esto puede limitar oportunidades de recompra, venta cruzada y aumento del valor de cada compra.

3. **¿Qué solución podemos construir y qué resultados debería entregar?**  
   Podemos construir un sistema de recomendación que utilice el historial de compra de cada cliente para estimar qué productos podrían resultarle relevantes. La salida esperada será una lista ordenada de productos recomendados para cada cliente.

4. **¿Quién utilizaría las recomendaciones generadas por el modelo?**  
   Las recomendaciones podrían ser utilizadas por la plataforma de e-commerce, por campañas de marketing o por equipos comerciales para personalizar la oferta mostrada a cada cliente.

5. **¿Cuáles son las KPI's del negocio que esperamos mejorar?**  
   Principalmente el ticket promedio por cliente y las ventas de productos recomendados. Más adelante será importante distinguir estos KPI's de negocio de las métricas técnicas con las que evaluaremos el modelo.

6. **¿Qué información podría ser útil para el objetivo?**  
   Historial de compras por cliente, frecuencia de compra, productos adquiridos, cantidades, gasto, recencia, productos comprados en una misma factura, popularidad de productos y comportamiento temporal. También sería útil contar con clics, visitas, productos mostrados y carritos abandonados, pero el dataset no incluye esa información.


## Librerías necesarias

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")



## Carga de datos

Comenzamos cargando el dataset y revisando sus primeras filas. En esta etapa todavía no transformaremos nada; primero necesitamos entender cómo viene estructurada la información original.


In [ ]:

ruta = "online_retail_II.csv"

df = pd.read_csv(
    ruta,
    encoding="ISO-8859-1"
)

df.head()


## Estructura general del dataset

In [ ]:

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

df.info()


In [ ]:

df.columns.tolist()



### Interpretación

El dataset se encuentra a nivel de línea de factura: una misma factura puede aparecer varias veces porque cada producto comprado ocupa un registro distinto. Esta estructura es importante porque posteriormente podremos reconstruir tanto el comportamiento de cada cliente como los productos que suelen comprarse juntos.


## Calidad de datos

### Valores nulos

In [ ]:

nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
}).sort_values("nulos", ascending=False)

nulos



### Interpretación

Los valores faltantes son especialmente importantes en `Customer ID`, porque los registros sin cliente identificado no pueden utilizarse directamente para construir recomendaciones personalizadas. Sin embargo, no conviene eliminarlos todavía: primero los conservaremos para describir el comportamiento general del negocio y posteriormente construiremos una versión limpia orientada al modelado.


### Registros duplicados

In [ ]:

duplicados = df.duplicated().sum()

print(f"Registros duplicados exactos: {duplicados:,}")
print(f"Porcentaje: {duplicados / len(df) * 100:.2f}%")



Los duplicados exactos deberán revisarse durante la limpieza. Si representan una repetición accidental del mismo registro, mantenerlos podría inflar artificialmente las cantidades vendidas, el gasto de los clientes y la frecuencia de compra.


### Tipos de datos

In [ ]:

df.dtypes



`InvoiceDate` debe convertirse a tipo fecha para poder estudiar temporalidad, recencia y frecuencia. Los identificadores como `Invoice`, `StockCode` y `Customer ID` deben tratarse como identificadores y no como variables numéricas continuas.


In [ ]:

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")


## Transacciones inválidas, cancelaciones y devoluciones

In [ ]:

resumen_calidad = pd.Series({
    "Quantity < 0": (df["Quantity"] < 0).sum(),
    "Quantity = 0": (df["Quantity"] == 0).sum(),
    "Price < 0": (df["Price"] < 0).sum(),
    "Price = 0": (df["Price"] == 0).sum(),
    "Facturas con C": df["Invoice"].astype(str).str.startswith("C").sum()
})

resumen_calidad


In [ ]:

cancelaciones = df[df["Invoice"].astype(str).str.startswith("C")].copy()
cancelaciones.head()



### Interpretación

Las cantidades negativas y las facturas cuyo identificador comienza con `C` representan principalmente cancelaciones o devoluciones. Estos registros son útiles para conocer problemas del negocio y comportamiento posterior a la compra, pero no deben interpretarse como preferencias positivas del cliente.

Para el modelo de recomendación necesitaremos una tabla de interacciones que represente compras reales, por lo que más adelante excluiremos cancelaciones, cantidades no positivas y precios no positivos.


## Variable de ingreso por línea

In [ ]:

df["Total"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "Total"]].describe()



El valor `Total` representa el importe de cada línea de factura y será una variable central para calcular ventas, gasto por cliente y ticket promedio.


## Dataset de ventas válidas

In [ ]:

df_ventas = df[
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (~df["Invoice"].astype(str).str.startswith("C"))
].copy()

print(f"Registros originales: {len(df):,}")
print(f"Registros de ventas válidas: {len(df_ventas):,}")
print(f"Registros excluidos: {len(df) - len(df_ventas):,}")



A partir de esta sección utilizaremos `df_ventas` para describir el comportamiento comercial. El dataframe original se conserva sin modificaciones destructivas para poder consultar posteriormente cancelaciones, devoluciones o problemas de calidad.


## Análisis temporal

In [ ]:

print("Primera transacción:", df_ventas["InvoiceDate"].min())
print("Última transacción:", df_ventas["InvoiceDate"].max())

df_ventas["Year"] = df_ventas["InvoiceDate"].dt.year
df_ventas["Month"] = df_ventas["InvoiceDate"].dt.month
df_ventas["YearMonth"] = df_ventas["InvoiceDate"].dt.to_period("M")
df_ventas["DayOfWeek"] = df_ventas["InvoiceDate"].dt.day_name()
df_ventas["Hour"] = df_ventas["InvoiceDate"].dt.hour


In [ ]:

ventas_mensuales = (
    df_ventas
    .groupby("YearMonth", observed=True)["Total"]
    .sum()
)

plt.figure(figsize=(12, 5))
ventas_mensuales.plot()
plt.title("Evolución mensual de las ventas")
plt.xlabel("Mes")
plt.ylabel("Ventas")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



### Interpretación

La evolución mensual permite identificar tendencias y posibles patrones estacionales. Si algunos periodos concentran una mayor demanda, la temporalidad puede ser útil más adelante para evitar que el recomendador dependa únicamente de popularidad histórica y pierda cambios recientes en el comportamiento de compra.


## Análisis geográfico

In [ ]:

ventas_por_pais = (
    df_ventas
    .groupby("Country")["Total"]
    .sum()
    .sort_values(ascending=False)
)

ventas_por_pais.head(10)


In [ ]:

plt.figure(figsize=(10, 6))
ventas_por_pais.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 países por ventas")
plt.xlabel("Ventas")
plt.ylabel("País")
plt.tight_layout()
plt.show()


In [ ]:

participacion_uk = ventas_por_pais.get("United Kingdom", 0) / ventas_por_pais.sum() * 100
print(f"Participación del Reino Unido en ventas: {participacion_uk:.2f}%")



### Interpretación

El análisis por país permite saber si el comportamiento de compra está concentrado en un solo mercado. Una concentración muy alta en Reino Unido implicaría que los patrones aprendidos por el modelo estarán dominados por ese mercado, por lo que el país deberá considerarse al interpretar las recomendaciones y posiblemente como parte de la estrategia de segmentación.


## Análisis de productos

In [ ]:

print(f"Productos únicos: {df_ventas['StockCode'].nunique():,}")
print(f"Descripciones únicas: {df_ventas['Description'].nunique():,}")


In [ ]:

productos_mas_vendidos = (
    df_ventas
    .groupby(["StockCode", "Description"], dropna=False)["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

productos_mas_vendidos


In [ ]:

productos_mayor_ingreso = (
    df_ventas
    .groupby(["StockCode", "Description"], dropna=False)["Total"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

productos_mayor_ingreso



### Interpretación

Los productos más vendidos por cantidad no necesariamente son los mismos que generan mayores ingresos. Esta diferencia será importante para el negocio: un recomendador puede optimizar relevancia para el cliente, pero también deberá monitorearse si favorece productos con mayor potencial económico.

La popularidad de producto también servirá posteriormente como **baseline**: si un modelo personalizado no supera una recomendación simple de productos populares, entonces la complejidad adicional no estaría aportando suficiente valor.


## Análisis de clientes

In [ ]:

df_clientes = df_ventas.dropna(subset=["Customer ID"]).copy()

print(f"Clientes únicos identificados: {df_clientes['Customer ID'].nunique():,}")
print(f"Registros con cliente identificado: {len(df_clientes):,}")


In [ ]:

facturas_por_cliente = (
    df_clientes
    .groupby("Customer ID")["Invoice"]
    .nunique()
)

facturas_por_cliente.describe()


In [ ]:

plt.figure(figsize=(10, 5))
plt.hist(facturas_por_cliente.clip(upper=facturas_por_cliente.quantile(0.99)), bins=40)
plt.title("Distribución del número de compras por cliente")
plt.xlabel("Número de facturas")
plt.ylabel("Número de clientes")
plt.tight_layout()
plt.show()


In [ ]:

clientes_recurrentes = (facturas_por_cliente >= 2).sum()
clientes_total = len(facturas_por_cliente)

print(f"Clientes con 2 o más compras: {clientes_recurrentes:,}")
print(f"Porcentaje de clientes recurrentes: {clientes_recurrentes / clientes_total * 100:.2f}%")



### Interpretación

La recurrencia es una de las condiciones más importantes para el sistema de recomendación. Un cliente con varias compras aporta historial suficiente para separar una parte de sus interacciones como entrenamiento y otra como validación.

Los clientes con una sola compra representan un problema de **cold start**: el sistema conoce muy poco sobre ellos. Para estos casos será conveniente disponer de una recomendación alternativa basada en popularidad, país o segmento.


## Gasto y ticket promedio

In [ ]:

gasto_por_cliente = (
    df_clientes
    .groupby("Customer ID")["Total"]
    .sum()
)

ticket_por_factura = (
    df_clientes
    .groupby(["Customer ID", "Invoice"])["Total"]
    .sum()
    .reset_index(name="Ticket")
)

ticket_por_cliente = (
    ticket_por_factura
    .groupby("Customer ID")["Ticket"]
    .mean()
)

resumen_clientes = pd.DataFrame({
    "facturas": facturas_por_cliente,
    "gasto_total": gasto_por_cliente,
    "ticket_promedio": ticket_por_cliente
})

resumen_clientes.describe()



El ticket promedio es un KPI de negocio, no una métrica directa del recomendador. El modelo será evaluado inicialmente con métricas offline de recomendación; el incremento real del ticket promedio requeriría posteriormente una validación en producción, idealmente mediante una prueba A/B.


## Relación cliente-producto

In [ ]:

interacciones = (
    df_clientes
    .groupby(["Customer ID", "StockCode"])
    .agg(
        compras=("Invoice", "nunique"),
        cantidad_total=("Quantity", "sum"),
        gasto_total=("Total", "sum"),
        ultima_compra=("InvoiceDate", "max")
    )
    .reset_index()
)

interacciones.head()


In [ ]:

n_clientes = df_clientes["Customer ID"].nunique()
n_productos = df_clientes["StockCode"].nunique()
n_interacciones = len(interacciones)

densidad = n_interacciones / (n_clientes * n_productos)
sparsity = 1 - densidad

print(f"Clientes: {n_clientes:,}")
print(f"Productos: {n_productos:,}")
print(f"Interacciones cliente-producto: {n_interacciones:,}")
print(f"Densidad de la matriz: {densidad:.6f}")
print(f"Sparsity de la matriz: {sparsity:.2%}")



### Interpretación

La matriz cliente-producto suele ser muy dispersa en sistemas de recomendación: cada cliente compra solamente una pequeña fracción del catálogo disponible.

Una sparsity alta no invalida el proyecto, pero sí afecta la elección del modelo. Métodos de filtrado colaborativo, factorización de matrices o modelos para feedback implícito están diseñados precisamente para trabajar con este tipo de información.


## Recompra de productos

In [ ]:

recompras = (interacciones["compras"] >= 2).sum()

print(f"Relaciones cliente-producto con recompra: {recompras:,}")
print(f"Porcentaje sobre todas las relaciones: {recompras / len(interacciones) * 100:.2f}%")



La recompra permite identificar productos que forman parte de hábitos recurrentes. Sin embargo, el recomendador no debería limitarse a repetir artículos ya comprados: también deberá aprender relaciones entre clientes y productos para favorecer descubrimiento y venta cruzada.


## Productos por factura y oportunidades de venta cruzada

In [ ]:

productos_por_factura = (
    df_clientes
    .groupby("Invoice")["StockCode"]
    .nunique()
)

productos_por_factura.describe()


In [ ]:

plt.figure(figsize=(10, 5))
plt.hist(productos_por_factura.clip(upper=productos_por_factura.quantile(0.99)), bins=40)
plt.title("Cantidad de productos distintos por factura")
plt.xlabel("Productos distintos")
plt.ylabel("Número de facturas")
plt.tight_layout()
plt.show()



### Interpretación

Las facturas con varios productos contienen información especialmente útil para venta cruzada, ya que permiten identificar productos que aparecen juntos en una misma compra. Esta estructura puede utilizarse posteriormente con reglas de asociación o similitud entre productos como complemento del recomendador principal.


## Popularidad y concentración del catálogo

In [ ]:

clientes_por_producto = (
    df_clientes
    .groupby("StockCode")["Customer ID"]
    .nunique()
    .sort_values(ascending=False)
)

clientes_por_producto.describe()


In [ ]:

productos_poco_frecuentes = (clientes_por_producto < 5).sum()

print(f"Productos comprados por menos de 5 clientes: {productos_poco_frecuentes:,}")
print(f"Porcentaje: {productos_poco_frecuentes / len(clientes_por_producto) * 100:.2f}%")



Los productos con muy pocas interacciones pueden dificultar el aprendizaje del modelo. Más adelante será necesario decidir si se conserva todo el catálogo o si se establece un mínimo de interacciones para el entrenamiento, procurando no eliminar demasiada variedad ni provocar underfitting.


## Variables candidatas para Feature Engineering


A partir del EDA podemos identificar varias variables útiles para representar el comportamiento de compra:

### A nivel de cliente
- Recency.
- Frequency.
- Monetary.
- Ticket promedio.
- Productos distintos comprados.
- Cantidad promedio por compra.
- Antigüedad del cliente.
- Intervalo promedio entre compras.
- País principal.

### A nivel de producto
- Número de clientes compradores.
- Cantidad total vendida.
- Ingreso total.
- Popularidad reciente.
- Tasa de devolución.
- Recencia de la última venta.

### A nivel cliente-producto
- Número de compras.
- Cantidad acumulada.
- Gasto acumulado.
- Última fecha de compra.
- Días desde la última interacción.
- Indicador de recompra.

No todas estas variables deben entrar en un mismo modelo. Primero evaluaremos cuáles aportan información real y cuáles son redundantes para evitar agregar complejidad sin beneficio.


## Conclusiones del EDA


El dataset contiene suficiente historial transaccional para desarrollar un sistema de recomendación basado en comportamiento de compra. La presencia de identificadores de cliente y producto permite construir una matriz de interacciones cliente-producto, mientras que las facturas permiten estudiar productos adquiridos conjuntamente.

El principal reto es que no todos los registros pueden utilizarse directamente. Existen cancelaciones, devoluciones, valores faltantes y clientes sin identificación, por lo que será necesario construir un dataset limpio orientado al modelado.

La recurrencia de compra será clave para validar el sistema. Los clientes con suficiente historial permitirán utilizar una división temporal entre compras pasadas y futuras, mientras que los clientes con poco historial requerirán una estrategia de cold start.

El clustering puede utilizarse para describir perfiles de comportamiento, pero el objetivo final del proyecto será evaluar un sistema que genere recomendaciones personalizadas. Por esta razón, además de métricas de segmentación, el modelo de recomendación deberá compararse contra un baseline y evaluarse posteriormente con métricas como Precision@K y Recall@K.

Finalmente, los KPI's de negocio como ticket promedio o ventas asociadas a recomendaciones servirán para comunicar el impacto esperado de la solución, pero no pueden atribuirse causalmente al modelo utilizando únicamente este historial de transacciones. Para comprobar ese impacto sería necesario realizar una validación posterior en producción.


## Decisiones para la siguiente etapa: ETL y modelado


Como resultado del EDA, la siguiente etapa deberá preparar al menos las siguientes estructuras:

- **Tabla de transacciones limpias:** compras válidas a nivel de línea de factura.
- **Tabla de facturas:** ticket total, cantidad de productos y cliente por compra.
- **Tabla de clientes:** variables RFM y comportamiento agregado.
- **Tabla de productos:** popularidad, ingresos e interacciones.
- **Tabla de interacciones cliente-producto:** base para el recomendador.
- **Tabla de recomendaciones:** cliente, producto, score, ranking y versión del modelo.

Estas estructuras permitirán separar correctamente el análisis descriptivo, la segmentación y el entrenamiento del recomendador.
